# Lot D — Silver officiel : verdict par LLM ouvert contraint aux preuves (EXP-D-010)

**Auteur : Membre 5 (Hadil).** Résultat **officiel** du Silver, conforme au cahier des charges
(modèle ouvert de **8 milliards de paramètres au plus** en inférence) et à la décision de l'équipe
(les données Tunifact ne sortent pas vers une API ; Gemini reste une comparaison, EXP-D-009).

- LLM : `CohereLabs/aya-expanse-8b` (8 Md, 23 langues dont l'arabe et le français), chargé en 4 bits ;
- même prompt contrôlé v1 et mêmes contrôles qu'EXP-D-009, recherche hybride (EXP-D-008) ;
- **3 vraies graines** (13, 42, 2026) : la graine est fixée localement, donc reproductible ;
- 4 ablations, baselines, bootstrap apparié.

Avant de commencer : *Exécution → Modifier le type d'exécution → **GPU T4***.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Accès au modèle (une seule fois)
Aya Expanse est à **accès contrôlé** : sur https://huggingface.co/CohereLabs/aya-expanse-8b, connecte-toi
et accepte la licence (usage non commercial). Puis crée un jeton *Read* sur
https://huggingface.co/settings/tokens. Le jeton se colle dans la case masquée ci-dessous,
**jamais dans une cellule**.

In [ ]:
import os
from getpass import getpass

os.environ["HF_TOKEN"] = getpass("Colle ton jeton Hugging Face (lecture) : ").strip()
print("✅ jeton Hugging Face enregistré pour cette session")

## 2. Récupérer le projet (branche `lot-d-silver`)
Jeton GitHub *classic* (case `repo`, 7 jours), dans la case masquée ; à supprimer sur GitHub après usage.

In [ ]:
import re, subprocess

token = getpass("Colle ton token GitHub puis appuie sur Entrée : ").strip()
url = f"https://{token}@github.com/SouissiOumaima/NewsCheckTN.git"
r = subprocess.run(["git", "clone", "-q", "-b", "lot-d-silver", url], capture_output=True, text=True)
del token, url
print("✅ Projet téléchargé" if r.returncode == 0
      else "❌ " + re.sub(r"gh[pousr]_\w+", "ghp_***", r.stderr))
%cd NewsCheckTN
!git remote set-url origin https://github.com/SouissiOumaima/NewsCheckTN.git
!git log --oneline -1

In [ ]:
!pip install -q -r requirements-dev.txt sentence-transformers bitsandbytes accelerate -U transformers

## 3. Corpus de test (privé, reconstruit dans la session, jamais poussé sur GitHub)

In [ ]:
!python -m rag_llm.build_tunifact_eval
!python -m rag_llm.freeze_test --check

## 4. Configuration

In [ ]:
os.environ.update({
    "LLM_PROVIDER": "hf",
    "LLM_MODEL": "CohereLabs/aya-expanse-8b",
    "RETRIEVAL_METHOD": "hybrid",
})
print({k: os.environ[k] for k in ("LLM_PROVIDER", "LLM_MODEL", "RETRIEVAL_METHOD")})

## 5. Essai rapide (3 affirmations du dev)
Le premier lancement télécharge le modèle (~16 Go, 5 à 10 min), puis vérifie que les sorties sont valides.

In [ ]:
!python -X utf8 -m rag_llm.silver_verdict --splits dev --limit 3

## 6. Dev complet (20 affirmations)
Regarder la colonne `valid.` et les raisons de rejet. **Envoyer ce résultat à Claude avant le test.**
Le prompt v1 est figé : on ne le change que si le format échoue sur le dev.

In [ ]:
!python -X utf8 -m rag_llm.silver_verdict --splits dev

## 7. Test gelé — une seule fois

In [ ]:
!python -X utf8 -m rag_llm.silver_verdict --splits test

## 8. Récupérer les résultats
Le zip contient les scores et les sorties du LLM (avec des extraits Tunifact : **privé**, à garder sur ton PC).
Si la session Colab se coupe avant la fin, télécharge quand même le cache : en le remettant dans
`evaluation/results/`, l'exécution reprend là où elle s'est arrêtée.

In [ ]:
!cd evaluation/results && zip -q ../../lot-d_silver_verdict_aya.zip lot-d_silver_verdict_*aya*.json lot-d_llm_cache.jsonl
from google.colab import files
files.download("lot-d_silver_verdict_aya.zip")